# Step 12 — RAG (Retrieval-Augmented Generation)

🇬🇧 **English** (this notebook)

RAG grounds a response in a document you provide: your text is chunked, embedded, and the most relevant chunks are retrieved and injected into the agent's context automatically. Unlike Steps 09–11, `knowledge_sources` does **not** work with a standalone `Agent.kickoff()` call — retrieval only gets wired up through a `Crew`'s own orchestration. So this one exercise needs a minimal inline `Crew` (still defined right here, no external YAML files) instead of a bare `Agent`.

## Learning objective

By the end of this notebook, you will:

- Understand what RAG (Retrieval-Augmented Generation) means, and what kind of information it's for that web search and MCP aren't
- Have pointed a `TextFileKnowledgeSource` at your own document and wired it into a `Crew`
- Be able to demonstrate, by removing the knowledge source, exactly what retrieval added to the answer

## Prerequisites

- [Step 10 — Tools](step_10_tools.ipynb) and [Step 11 — MCP](step_11_mcp.ipynb) completed — this notebook directly compares against both
- A `GEMINI_API_KEY` in your `.env`, even if your chat model (`MODEL`) is a different provider — embeddings are configured to use Gemini explicitly here
- The same `.env` setup as the previous steps otherwise

## Background

Retrieval-Augmented Generation grounds a model's answer in a specific document you provide, instead of relying on what it memorized during training or what a live web search happens to surface:

> Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)

## How this works

RAG isn't limited to one file or one format — a `Crew` can retrieve across several knowledge sources at once. This notebook wires up two, both about the same EU AI Act topic but different in kind:

1. `TextFileKnowledgeSource(file_paths=["ai_act_internal_review.txt"])` — a private internal document (included in this repo) with compliance gaps for one specific system. No web search or URL fetch could ever find this, because it was never published.
2. `PDFKnowledgeSource(file_paths=["EU AI Act Summary.pdf"])` — a public reference PDF summarizing the Act itself, including which use cases (like recruitment/selection) count as high-risk. Under the hood this uses `pdfplumber` (a direct `crewai` dependency, already installed) to extract each page's text before chunking — everything downstream (chunking, embedding, retrieval) works the same as the `.txt` source.
3. Both resolve `knowledge/` relative to wherever the notebook's kernel is running (this repo's `exercises/en/knowledge/`, by default) — not the repo root.
4. The agent's identity is once again the same Researcher template.
5. `Crew(..., knowledge_sources=[...], embedder={...})` is the minimum needed to activate retrieval — this is why RAG needs a `Crew` even for a single agent, unlike Steps 09–11. Pass a list with both sources and the agent can draw on either (or both) when answering.
6. The question needs *both* sources to answer fully: the general "is this kind of system high-risk" classification lives in the public PDF, while the specific compliance gaps for our system live in the private internal notes.
7. `tracing=True` — introduced in [Step 02](step_02_intro_to_crewai.ipynb) — prints a shareable dashboard link once the cell finishes; its timeline is a quick way to see the retrieval step actually happening, not just the final grounded answer.

In [ ]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource

load_dotenv()

# CrewAI's Gemini embedder reads its model name from an env var whose generic
# alias ("model") collides with this project's own MODEL variable — without
# this, retrieval silently fails (only visible with Crew(verbose=True)) and
# the agent falls back to guessing instead of grounding in your documents.
os.environ["EMBEDDINGS_GOOGLE_GENERATIVE_AI_MODEL_NAME"] = "gemini-embedding-001"

# ── Your knowledge documents — one private, one public, same topic ───────────
text_knowledge_source = TextFileKnowledgeSource(file_paths=["ai_act_internal_review.txt"])
pdf_knowledge_source = PDFKnowledgeSource(file_paths=["EU AI Act Summary.pdf"])

topic = "EU AI Act"

# ── Agent — same "researcher" template as agents.yaml, {topic} filled in via an f-string ─
role      = f"{topic} Senior Data Researcher"
goal      = f"Uncover cutting-edge developments in {topic}"
backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)
agent = Agent(role=role, goal=goal, backstory=backstory, verbose=True)

# ── Task ───────────────────────────────────────────────────────────────────────
# The "only state what's retrieved" instruction matters here: without it, a
# capable model will confidently fill gaps with plausible-sounding invented
# specifics even when no knowledge_sources are provided at all (try step 3
# below) — which would quietly defeat the whole point of this comparison.
question = (
    "According to the EU AI Act, is our Resume Triage Assistant classified as "
    "high-risk, and according to our internal review notes, what compliance "
    "gaps remain? Only state facts you can find in the provided knowledge "
    "sources — if you can't find the answer there, say so explicitly instead "
    "of guessing."
)
task = Task(
    description=question,
    expected_output="A direct answer grounded in the EU AI Act summary and the internal review notes.",
    agent=agent,
)

# ── Crew ─────────────────────────────────────────────────────────────────────
crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    knowledge_sources=[text_knowledge_source, pdf_knowledge_source],
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes.
    tracing=True,
    embedder={
        "provider": "google-generativeai",
        "config": {"api_key": os.getenv("GEMINI_API_KEY"), "model_name": "gemini-embedding-001"},
    },
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print(result.raw)

## Your task

1. Run the cell. Does the agent correctly combine both sources — the high-risk classification from the public `EU AI Act Summary.pdf` and the compliance gaps from the private `ai_act_internal_review.txt`?

2. Compare to Steps 10/11: could a web search or a URL fetch ever have answered the internal-notes half of this question? What kind of information is RAG for that tools/MCP aren't?

3. Now pass an empty list instead (`knowledge_sources=[]`) and rerun. Does the agent still answer correctly, or does it admit it doesn't know? That comparison is the point of RAG.

4. Add your own `.txt` or `.pdf` file to `knowledge/` with background on your team's topic, and swap in your own agent identity and question.

5. Note what you observed — it's evidence for `REPORT.md`'s Section 4.2 (Memory) and Section 4.4 (Context Engineering: Context Retrieval). This closes out Sprint 3 — add its row to the **Sprint Progression** table too.

## Shortcomings

Retrieval only surfaces what's already written down in the document you provided — it can't reason across documents it wasn't given, and a poorly written or outdated source document produces a confidently-grounded *wrong* answer just as easily as a correct one. Chunking also loses some context: a fact split across two chunk boundaries may never be retrieved together.

Everything from Step 09 through here has also been **one agent** doing everything itself — the same Researcher that reasons, searches, fetches, and now retrieves is also the one who'd have to turn all of that into a report for a specific audience. [Step 13](step_13_multi_agent_seq.ipynb) closes that gap: splitting research and reporting across two complementary agents.

## Resources for further reading

- Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)
- [CrewAI Knowledge concept docs](https://docs.crewai.com/en/concepts/knowledge)

## Stretch goal

Upload your own PDF instead of using `EU AI Act Summary.pdf` — a policy doc, a paper, a manual, anything with a fact tucked away on a specific page:

```python
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
your_pdf_source = PDFKnowledgeSource(file_paths=["your_document.pdf"])
```

Drop the file into `knowledge/`, swap it into `knowledge_sources=[...]`, and ask a question whose answer only appears on that specific page. Does the agent retrieve it?